<a href="https://colab.research.google.com/github/crezny/DATASCI266_final_project/blob/main/atomic_text_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install evaluate -q
%pip install rouge_score -q
%pip install sentence_transformers -q
%pip install transformers -q

In [2]:
import pandas as pd
import evaluate
import re
import os
import numpy as np
from rouge_score import rouge_scorer, scoring
from sentence_transformers import SentenceTransformer
from typing import List, Dict, Optional, Tuple, Any, Sequence
from transformers import AutoModelForCausalLM, AutoTokenizer,AutoModelForSequenceClassification
import json

import matplotlib.pyplot as plt
import math

try:
    from tqdm.notebook import tqdm
except Exception:
    from tqdm import tqdm


from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError
try:
    from google.colab import userdata
    get_secret = userdata.get
except ImportError:
    # Fallback: define a dummy get_secret for local use
    def get_secret(key):
        return None

def get_env(key: str, default: str = None):
    """Tries os.getenv first, then Colab userdata if available."""
    return os.getenv(key) or get_secret(key) or default


DB_HOST = get_env("POSTGRES_DB_HOST")
DB_PORT = int(get_env("POSTGRES_DB_PORT"))
DB_NAME = get_env("POSTGRES_DB_NAME")
DB_USER = get_env("POSTGRES_DB_USER")
DB_PASS = get_env("POSTGRES_DB_PASS")

connection_url = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_url, echo=False)

from transformers import RobertaTokenizer, RobertaModel
import torch
from sentence_transformers import SentenceTransformer, util, models

In [3]:
subcontrols_sql = """
SELECT
    subcontrols.id   AS subcontrol_id,
    subcontrols.title AS subcontrol_name,
    control_areas.id  AS control_area_id,
    control_areas.name AS control_area_name,
	control_areas.description AS control_area_description,
    frameworks.id      AS frameworks_id,
    frameworks.name    AS frameworks_name,
    overviews.description as overview
FROM subcontrols
JOIN control_areas ON subcontrols.control_id = control_areas.id
left outer join overviews on subcontrols.id = overviews.subcontrol_id
JOIN frameworks    ON control_areas.framework_id = frameworks.id
"""

subcontrols_df = pd.read_sql(text(subcontrols_sql), engine)

action_item_sql = """
SELECT
    REPLACE(id, CAST(subcontrol_id AS varchar(255)) || '_AI', '') AS ai_order,
    *
FROM action_items
ORDER BY subcontrol_id,
         REPLACE(id, CAST(subcontrol_id AS varchar(255)) || '_AI', '')
"""

action_items_df = pd.read_sql(text(action_item_sql), engine)

additional_guidence_sql = """
SELECT
    REPLACE(id, CAST(subcontrol_id AS varchar(255)) || '_AG', '') AS ag_order,
    *
FROM additional_guidances
ORDER BY subcontrol_id,
         REPLACE(id, CAST(subcontrol_id AS varchar(255)) || '_AG', '')

         """
additional_guidances_df = pd.read_sql(text(additional_guidence_sql), engine)

In [4]:
def split_description(desc: str):
    if not isinstance(desc, str):
        return []
    desc = desc.replace("\r\n", "\n").strip()
    if not desc:
        return []

    pieces = []

    # 1) split on blank lines / line breaks
    for chunk in re.split(r"\n+", desc):
        chunk = chunk.strip()
        if not chunk:
            continue

        # 2) inside each chunk, split on numbered list markers like "1)", "2.", "3 -"
        parts = re.split(r"\s*(?=\d+[\)\.\-]\s)", chunk)
        for part in parts:
            part = part.strip()
            if not part:
                continue

            # 3) remove leading numbers ("1)", "2.", "3 -", etc.)
            cleaned = re.sub(r"^\d+[\)\.\-]\s*", "", part).strip()
            if cleaned:
                pieces.append(cleaned)

    return pieces


In [5]:
# list of lines per action_item
action_items_df["ai_lines"] = action_items_df["description"].apply(split_description)

action_lists = (
    action_items_df
    .sort_values(["subcontrol_id", "ai_order"])  # keep action_item order
    .groupby("subcontrol_id")["ai_lines"]
    .apply(lambda rows: [line for lst in rows for line in lst])  # flatten list-of-lists
    .reset_index(name="action_items")
)

In [6]:
additional_guidances_df["ag_lines"] = additional_guidances_df["description"].apply(split_description)

guidance_lists = (
    additional_guidances_df
    .sort_values(["subcontrol_id", "ag_order"])
    .groupby("subcontrol_id")["ag_lines"]
    .apply(lambda rows: [line for lst in rows for line in lst])
    .reset_index(name="additional_guidance")
)

In [7]:
subcontrols_enriched = (
    subcontrols_df
    .merge(action_lists,   on="subcontrol_id", how="left")
    .merge(guidance_lists, on="subcontrol_id", how="left")
)

# optional: replace NaN with empty lists
for col in ["action_items", "additional_guidance"]:
    subcontrols_enriched[col] = subcontrols_enriched[col].apply(
        lambda x: x if isinstance(x, list) else []
    )

In [19]:
def create_atomic_prompt(
    control_area_description: str = '',
    overview_text: str = '',
    additional_guidance_text: list = [],
    action_items_text: list = []
):
    base_prompt = f"""
Here is the Control Area description for context:
<<<CONTROL_AREA>>>
{control_area_description.strip()}
<<<END_CONTROL_AREA>>>

Here is the subcontrol content:
<<<CONTENT>>>
""".strip()

    # Build clean content block with NO labels
    blocks = []

    if overview_text and overview_text.strip():
        blocks.append(overview_text.strip())

    if additional_guidance_text:
        blocks.extend([line.strip() for line in additional_guidance_text if line.strip()])

    if action_items_text:
        blocks.extend([line.strip() for line in action_items_text if line.strip()])

    combined_content = "\n".join(blocks)

    final_prompt = (
        base_prompt
        + "\n"
        + combined_content
        + "\n<<<END>>>"
    )

    return final_prompt


In [20]:
subcontrols_enriched["atomic_prompt"] = subcontrols_enriched.apply(
    lambda row: create_atomic_prompt(
        overview_text=row["overview"],
        additional_guidance_text=row["additional_guidance"],
        action_items_text=row["action_items"],
        control_area_description=row["control_area_description"]
    ),
    axis=1
)

In [10]:
#@title load model
model_name = "Qwen2.5-7B-Instruct"
model_path =  "Qwen/Qwen2.5-7B-Instruct"
model_path: str
prompt_col: str = "prompt_text"
system_prompt: str = ""
max_new_tokens: int = 256
temperature: float = 0.2
top_p: float = 0.9

tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)
torch_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    dtype=torch_dtype,
    device_map="auto",
)
model.eval()



Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)
    (ro

In [16]:
subcontrols_enriched.iloc[0]

,0
subcontrol_id,9292378
subcontrol_name,Roles and Responsibilities (ID.AM-6)
control_area_id,69870
control_area_name,Asset Management (ID.AM)
control_area_description,"The data, personnel, devices, systems, and fac..."
frameworks_id,5
frameworks_name,NIST CSF
overview,None
action_items,[Verify this control requirement is documented...
additional_guidance,[]


In [93]:
system_msg = """You are helping normalize cybersecurity control text.

You will receive a single block of raw text that may contain:
- high-level descriptions,
- contextual guidance,
- or procedural / action-item language.

The block may contain any subset of these.
Your task is to extract ONLY the underlying ORGANIZATIONAL REQUIREMENTS.

OUTPUT RULES (STRICT):
- Output ONLY a JSON array of strings. No other text.
- Output MUST be valid JSON.
- Do NOT include headings, labels, explanations, or comments.
- Do NOT summarize the text; only extract obligations that are explicitly implied.
- If the text contains no organizational obligations, output: [].

REQUIREMENT RULES:
- Each requirement must describe what the organization, its personnel, or its systems MUST do.
- Remove audit language such as: "verify that", "review", "inspect",
  "interview personnel", "through observation", "by reviewing documentation", etc.
- Convert audit steps into the obligations they imply.
- Keep the original meaning and scope. Do NOT add new requirements.
- Split multi-part obligations into separate sentences when feasible.
- Each requirement should be one clear sentence, about 10–25 words.
- Use direct normative language:
  - "The organization must ..."
  - "Personnel must ..."
  - "Systems must ...".
- Do NOT mention audits, testing, observations, evidence, or verification.

CONTENT SAFEGUARDS:
- Do NOT treat descriptive, contextual, background, or informational statements as obligations.
- Do NOT convert examples or explanatory details into obligations.
- Do NOT infer new obligations from capabilities or descriptions unless explicitly implied.
- Only extract obligations the text directly requires.

CONTROL AREA (IF PROVIDED):
- You may receive a Control Area description in addition to the subcontrol text.
- Use it ONLY for context.
- Do NOT create requirements based on the Control Area description.
- Do NOT summarize or restate the Control Area.
- Do NOT assume obligations from it.

EXAMPLE:
Input:
<<<CONTENT>>>
Verify that all remote access sessions are logged.
Interview administrators to confirm they review remote access logs weekly.
<<<END>>>

Correct JSON output:
[
  "The organization must log all remote access sessions.",
  "Administrators must review remote access logs at least weekly."
]
"""


In [96]:
def run_qwen(messages: list = [], max_new_tokens: int = 256):

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    enc = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        gen_ids = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            temperature=None,
            top_k=None,
            do_sample=False,
            top_p=1.0,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # strip prompt tokens
    gen_only_ids = gen_ids[0][enc["input_ids"].shape[1]:]
    raw_text = tokenizer.decode(gen_only_ids, skip_special_tokens=True).strip()

    # safety net: extract first JSON array in the text
    import re
    match = re.search(r"\[[\s\S]*\]", raw_text)
    if not match:
        raise ValueError(f"No JSON array found in model output:\n{raw_text}")

    json_str = match.group(0)
    try:
        data = json.loads(json_str)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON from model:\n{json_str}") from e

    return data

In [38]:
print(subcontrols_enriched.iloc[0]['action_items'])

['Verify this control requirement is documented and known by all parties.', 'Verify that information security policies clearly define information security responsibilities for all personnel.', 'Interview a sample of responsible personnel to verify they understand the security policies.']


In [94]:
user_msg = subcontrols_enriched.iloc[0]['atomic_prompt']

messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ]

In [95]:
gen_text = run_qwen(messages, 512)



[
  "The organization must document this control requirement.",
  "Information security policies must clearly define information security responsibilities for all personnel.",
  "Personnel must understand the security policies."
]


In [97]:
print('\n'.join(gen_text))

The organization must document this control requirement.
Information security policies must clearly define information security responsibilities for all personnel.
Personnel must understand the security policies.


In [98]:
results = []

for row in tqdm(
    subcontrols_enriched.itertuples(index=False),
    total=len(subcontrols_enriched),
    desc="Extracting atomic requirements"
):
    # Pull fields from row (adjust names to match your DF)
    subcontrol_id = row.subcontrol_id

    control_area_description = getattr(row, "control_area_description", "")
    overview_text = getattr(row, "overview_text", "")
    additional_guidance_text = getattr(row, "additional_guidance", [])
    action_items_text = getattr(row, "action_items", [])

    # Build user prompt with your helper
    user_prompt = create_atomic_prompt(
        control_area_description=control_area_description,
        overview_text=overview_text,
        additional_guidance_text=additional_guidance_text,
        action_items_text=action_items_text,
    )

    # Build messages for Qwen2.5
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_prompt},
    ]

    try:
        atomic_reqs = run_qwen(messages, max_new_tokens=256)  # list of strings
    except Exception as e:
        # Up to you: log it, store empty, etc.
        atomic_reqs = []

    for idx, req in enumerate(atomic_reqs, start=1):
        results.append(
            {
                "subcontrol_id": subcontrol_id,
                "atomic_order": idx,
                "atomic_requirement": req,
            }
        )

# Turn into a DataFrame
atomic_requirements_df = pd.DataFrame(results)


Extracting atomic requirements:   0%|          | 0/170 [00:00<?, ?it/s]

In [103]:
import re

def is_low_value_requirement(text: str) -> bool:
    t = text.lower().strip()

    # 1. Generic references to "this control requirement"
    if "this control requirement" in t:
        return True

    # 2. Generic references to "the control requirement" (unnamed)
    if "the control requirement" in t:
        return True

    # 3. Vague "aware of" statements with no actual action
    #    (We only remove when it's not about a concrete policy/procedure)
    if re.search(r"\baware of (this|the) control requirement\b", t):
        return True

    # 4. Meta "must be known" or "must be communicated"
    #    when no object is defined
    if re.search(r"\bmust be (known|communicated)\b", t):
        if "procedure" not in t and "policy" not in t:
            return True

    # 5. Generic "document" statements with no object
    if t == "the organization must document this control requirement.":
        return True

    # 6. Very short, meaningless requirements
    if len(t.split()) < 5:
        return True

    return False


def filter_atomic_requirements(reqs):
    return [r for r in reqs if not is_low_value_requirement(r)]


In [105]:
atomic_requirements_df["is_low_value"] = atomic_requirements_df["atomic_requirement"].apply(
    is_low_value_requirement
)

In [110]:
atomic_requirements_df.to_sql(name='subcontrol_atomic_req', con=engine, if_exists='replace', index=True, index_label = 'id')

736